# Notebook 02: ML Development

**Persona:** Data Scientist

**Purpose:** Demonstrate the complete ML development workflow using Feature Store features:
1. Feature discovery and exploration
2. Training data generation with two approaches:
   - `generate_dataset` with SQL-level spine sampling (Conversion model)
   - `generate_training_set` with Python-level DataFrame sampling (Churn model)
3. Preprocessing including wide OBJECT unpacking
4. Model training (GradientBoosting for both models)
5. Model Registry persistence with lineage

**Prerequisites:** Run Notebooks 00 and 01 first

## 1. Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    sys.path.insert(0, ".")
    from feature_definitions.config import get_session, ROLES
    session = get_session(role=ROLES["dev"])

sys.path.insert(0, ".") if "." not in sys.path else None

from feature_definitions.config import get_config, ROLES
from snowflake.ml.feature_store import FeatureStore, CreationMode

cfg = get_config("DEV")
session.sql(f"USE ROLE {ROLES['dev']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

fs = FeatureStore(
    session=session,
    database=cfg["database"],
    name=cfg["fs_schema"],
    default_warehouse=cfg["warehouse"],
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print(f"Feature Store ready: {cfg['database']}.{cfg['fs_schema']}")

## 2. Feature Discovery

Before building models, explore what features are available. The Feature Store provides APIs for listing entities, feature views, and inspecting metadata.

In [ ]:
# List all entities
entities = fs.list_entities().to_pandas()
print("=== Entities ===")
for _, e in entities.iterrows():
    print(f"  {e['NAME']:25s}  keys={e['JOIN_KEYS']}")

# List all feature views
print("\n=== Feature Views ===")
fvs = fs.list_feature_views().to_pandas()
for _, fv in fvs.iterrows():
    print(f"  {fv['NAME']:40s} {fv['VERSION']:5s}  refresh={fv['REFRESH_FREQ']}")

In [ ]:
# Deep-dive into a specific Feature View
fv = fs.get_feature_view("USER_PURCHASE_AGGREGATES", "V03")
print(f"Name: {fv.name}")
print(f"Description: {fv.desc}")
print(f"Entities: {[e.name for e in fv.entities]}")
print(f"Timestamp column: {fv.timestamp_col}")
print(f"Refresh frequency: {fv.refresh_freq}")
print(f"\nSample output:")
fv.feature_df.limit(3).show()

## 3. Conversion Model: generate_dataset with SQL Sampling

### 3.1 Build the Spine

The spine defines *which entities at which points in time* to retrieve features for. For conversion prediction, each row is a session with a binary label.

We use `TABLESAMPLE BERNOULLI` at the SQL level for efficient sampling.

In [ ]:
db = cfg["database"]
src = cfg["source_schema"]
spines = cfg["spines_schema"]

# Create conversion spine: one row per identified session
session.sql(f"""
    CREATE OR REPLACE TABLE {db}.{spines}.CONVERSION_SPINE AS
    SELECT
        SESSION_ID,
        USER_ID,
        SESSION_START_TS AS LABEL_TS,
        CASE WHEN IS_CONVERTED THEN 1 ELSE 0 END AS IS_CONVERTED
    FROM {db}.{src}.SESSIONS
    WHERE USER_ID IS NOT NULL
""").collect()

total = session.sql(f"SELECT COUNT(*) AS N FROM {db}.{spines}.CONVERSION_SPINE").collect()[0]["N"]
pos = session.sql(f"SELECT COUNT(*) AS N FROM {db}.{spines}.CONVERSION_SPINE WHERE IS_CONVERTED=1").collect()[0]["N"]
print(f"Conversion spine: {total} sessions, {pos} converted ({100*pos/total:.1f}%)")

### 3.2 Generate Dataset

`generate_dataset` returns a materialised `Dataset` object with built-in versioning. The spine is sampled using SQL `TABLESAMPLE` *before* the join -- this is efficient for large datasets because filtering happens in the warehouse.

In [ ]:
# SQL-level sampling
spine_df = session.sql(f"""
    SELECT * FROM {db}.{spines}.CONVERSION_SPINE
    TABLESAMPLE BERNOULLI (80)
""")
print(f"Sampled spine: {spine_df.count()} rows (from {total})")

# Retrieve Feature Views for conversion model
fv_session  = fs.get_feature_view("SESSION_BEHAVIOR_FEATURES", "V01")
fv_profile  = fs.get_feature_view("USER_PROFILE_FEATURES", "V01")
fv_purchase = fs.get_feature_view("USER_PURCHASE_AGGREGATES", "V03")
fv_engage   = fs.get_feature_view("USER_SESSION_ENGAGEMENT", "V01")

# generate_dataset – materialises a versioned Dataset
ds = fs.generate_dataset(
    name="CONVERSION_TRAINING",
    version="V01",
    spine_df=spine_df,
    features=[fv_session, fv_profile, fv_purchase, fv_engage],
    spine_timestamp_col="LABEL_TS",
    spine_label_cols=["IS_CONVERTED"],
    output_type="dataset",
    join_method="cte",
    desc="Conversion model training data with SQL-sampled spine",
)
print(f"\nDataset: {ds.fully_qualified_name}")

In [ ]:
# Read back and inspect
ds_df = ds.read.to_snowpark_dataframe()
conv_pd = ds_df.to_pandas()
print(f"Shape: {conv_pd.shape}")
print(f"\nColumns: {list(conv_pd.columns)}")
print(f"\nLabel distribution:\n{conv_pd['IS_CONVERTED'].value_counts()}")
conv_pd.head(3)

## 4. Churn Model: generate_training_set with Python Sampling

### 4.1 Build the Spine

For churn prediction, each row is a user with a binary label (churned = inactive).

In [ ]:
# Create churn spine
session.sql(f"""
    CREATE OR REPLACE TABLE {db}.{spines}.CHURN_SPINE AS
    SELECT
        USER_ID,
        UPDATED_TS AS LABEL_TS,
        CASE WHEN IS_ACTIVE THEN 0 ELSE 1 END AS IS_CHURNED
    FROM {db}.{src}.USERS
""").collect()

churn_total = session.sql(f"SELECT COUNT(*) AS N FROM {db}.{spines}.CHURN_SPINE").collect()[0]["N"]
churned = session.sql(f"SELECT SUM(IS_CHURNED) AS N FROM {db}.{spines}.CHURN_SPINE").collect()[0]["N"]
print(f"Churn spine: {churn_total} users, {churned} churned ({100*churned/churn_total:.1f}%)")

### 4.2 Generate Training Set

`generate_training_set` returns a Snowpark DataFrame (not materialised). Sampling is done *after* the join using Python's `pandas.DataFrame.sample()` -- useful when you want to inspect the full result before sampling.

In [ ]:
spine_df = session.table(f"{db}.{spines}.CHURN_SPINE")

# Shared + churn-specific FVs
fv_profile  = fs.get_feature_view("USER_PROFILE_FEATURES", "V01")
fv_purchase = fs.get_feature_view("USER_PURCHASE_AGGREGATES", "V03")
# Use USER_RECENCY_RAW (raw timestamps) for PIT-correct training data.
# RFM components come from existing FVs — no separate RFM FV needed.
fv_recency_raw = fs.get_feature_view("USER_RECENCY_RAW", "V01")
fv_trend       = fs.get_feature_view("USER_TREND_FEATURES", "V01")

# generate_training_set – returns DataFrame (lazy)
training_df = fs.generate_training_set(
    spine_df=spine_df,
    features=[fv_profile, fv_purchase, fv_recency_raw, fv_trend],
    spine_timestamp_col="LABEL_TS",
    spine_label_cols=["IS_CHURNED"],
    join_method="cte",
)
print(f"Training set columns: {training_df.columns}")
print(f"Training set rows: {training_df.count()}")

# Python-level sampling
churn_pd = training_df.to_pandas()
churn_sample = churn_pd.sample(frac=0.8, random_state=42)

# Post-retrieval PIT-correct enrichment (Ch 06 best practice)
# Derives DAYS_SINCE_LAST_ORDER, DAYS_SINCE_LAST_SESSION, etc.
# from raw timestamps relative to the spine's LABEL_TS.
from feature_definitions.enrichment import derive_temporal_features
churn_sample = derive_temporal_features(churn_sample, spine_ts_col="LABEL_TS")

print(f"\nAfter Python sampling + enrichment (80%): {len(churn_sample)} rows")
print(f"Enriched columns: {list(churn_sample.columns)}")
print(f"Label distribution:\n{churn_sample['IS_CHURNED'].value_counts()}")

### 4.3 Comparing the Two Approaches

| Aspect | `generate_dataset` | `generate_training_set` |
|---|---|---|
| **Returns** | `Dataset` object (materialised) | Snowpark DataFrame (lazy) |
| **Versioning** | Built-in (name + version) | Manual |
| **Sampling** | SQL-level (`TABLESAMPLE`) before join | Python-level (`.sample()`) after join |
| **Best for** | Reproducible experiments, large datasets | Interactive exploration, quick iteration |
| **Storage** | Persisted as Snowflake Dataset | Ephemeral unless saved |

## 5. Preprocessing

### Conversion Model

In [ ]:
import pandas as pd
import numpy as np

# Select numeric features, handle NaN
numeric_cols = conv_pd.select_dtypes(include=["number"]).columns.tolist()
label_col = "IS_CONVERTED"
conv_pd[label_col] = conv_pd[label_col].astype(int)

exclude = [label_col, "SESSION_ID", "USER_ID"]
feature_cols_conv = [c for c in numeric_cols if c not in exclude]
print(f"Conversion features ({len(feature_cols_conv)}): {feature_cols_conv}")

X_conv = conv_pd[feature_cols_conv].fillna(0)
y_conv = conv_pd[label_col]
print(f"\nX shape: {X_conv.shape}, y distribution: {dict(y_conv.value_counts())}")

### Churn Model

In [ ]:
# For churn, also numeric features
numeric_cols_churn = churn_sample.select_dtypes(include=["number"]).columns.tolist()
label_col_churn = "IS_CHURNED"

exclude_churn = [label_col_churn, "USER_ID"]
feature_cols_churn = [c for c in numeric_cols_churn if c not in exclude_churn]
print(f"Churn features ({len(feature_cols_churn)}): {feature_cols_churn}")

X_churn = churn_sample[feature_cols_churn].fillna(0)
y_churn = churn_sample[label_col_churn]
print(f"\nX shape: {X_churn.shape}, y distribution: {dict(y_churn.value_counts())}")

## 6. Model Training

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# --- Conversion Model ---
print("=== Conversion Model (GBM) ===")
try:
    Xtr, Xte, ytr, yte = train_test_split(X_conv, y_conv, test_size=0.2, random_state=42, stratify=y_conv)
except ValueError:
    Xtr, Xte, ytr, yte = train_test_split(X_conv, y_conv, test_size=0.2, random_state=42)

conv_model = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
conv_model.fit(Xtr, ytr)
y_pred = conv_model.predict_proba(Xte)
if y_pred.shape[1] > 1:
    try:
        auc = roc_auc_score(yte, y_pred[:, 1])
        print(f"  AUC: {auc:.4f}")
    except ValueError:
        print("  AUC: insufficient classes in test set")
print(f"  Train: {len(Xtr)}, Test: {len(Xte)}")

# Feature importance
fi = pd.Series(conv_model.feature_importances_, index=feature_cols_conv).sort_values(ascending=False)
print(f"\n  Top 5 features:")
for feat, imp in fi.head(5).items():
    print(f"    {feat:35s} {imp:.4f}")

In [ ]:
# --- Churn Model ---
print("=== Churn Model (GBM) ===")
try:
    Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn)
except ValueError:
    Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42)

churn_model = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
churn_model.fit(Xtr_c, ytr_c)
y_pred_c = churn_model.predict_proba(Xte_c)
if y_pred_c.shape[1] > 1:
    try:
        auc_c = roc_auc_score(yte_c, y_pred_c[:, 1])
        print(f"  AUC: {auc_c:.4f}")
    except ValueError:
        print("  AUC: insufficient classes in test set")
print(f"  Train: {len(Xtr_c)}, Test: {len(Xte_c)}")

fi_c = pd.Series(churn_model.feature_importances_, index=feature_cols_churn).sort_values(ascending=False)
print(f"\n  Top 5 features:")
for feat, imp in fi_c.head(5).items():
    print(f"    {feat:35s} {imp:.4f}")

## 7. Model Registry

Persist both models to the Snowflake Model Registry. This captures the model artifact, a sample of input data, and provides lineage tracking.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name=cfg["database"],
    schema_name=cfg["ml_datasets_schema"],
)

# Log conversion model
mv_conv = registry.log_model(
    model=conv_model,
    model_name="CONVERSION_PREDICTION",
    version_name="V01",
    sample_input_data=Xtr.head(10),
    comment="GBM conversion prediction – e2e demo (NB02)",
)
print(f"Logged: {mv_conv.model_name} / {mv_conv.version_name}")

# Log churn model
mv_churn = registry.log_model(
    model=churn_model,
    model_name="CHURN_PREDICTION",
    version_name="V01",
    sample_input_data=Xtr_c.head(10),
    comment="GBM churn prediction – e2e demo (NB02)",
)
print(f"Logged: {mv_churn.model_name} / {mv_churn.version_name}")

# Show all models
print("\n=== Models in Registry ===")
models = registry.show_models()
for _, row in models.iterrows():
    print(f"  {row['name']}")

## Summary

In this notebook we:

1. **Discovered** 13 Feature Views and 8 Entities in the Feature Store
2. **Generated training data** using two approaches:
   - `generate_dataset` with SQL `TABLESAMPLE` for the Conversion model
   - `generate_training_set` with Python `.sample()` for the Churn model
3. **Trained** GBM classifiers for both use cases, demonstrating shared and distinct features
4. **Persisted** both models to the Snowflake Model Registry

Proceeed to **Notebook 03: Model Deployment** for promotion, batch inference, and online serving.